
## File 03 — Job Scheduling and Monitoring:


This notebook prepares monitoring for scheduled Databricks pipeline executions.

We'll cover:

- Job scheduling strategy.
- Pipeline execution status.
- Task success and failure monitoring.
- Execution duration.
- Failed-task investigation.
- Job run history.
- Monitoring reports in Delta tables.


## Pipeline Schedule


| Pipeline | Suggested Schedule | Purpose |
|---|---|---|
| Bronze Ingestion | Every 1 hour | Detect and ingest new CSV files using Auto Loader |
| Silver Processing | After Bronze | Standardize, clean, deduplicate, and MERGE new records |
| Data Quality | After Silver | Validate transformed data and business rules |
| Referential Integrity | After Data Quality | Validate relationships between parent and child tables |
| Gold Refresh | After Validation | Refresh analytics tables and business metrics |
| Final Validation | Last task | Verify pipeline outputs and final execution status |


## Pipeline Execution Flow


```text
+----------------------------------+
| 01. BRONZE INGESTION             |
| Schedule: Every 1 hour           |
| Detect and ingest new CSV files  |
+----------------------------------+
                 |
                 v
+----------------------------------+
| 02. SILVER PROCESSING            |
| Schedule: After Bronze           |
| Clean, standardize and MERGE     |
| new records                      |
+----------------------------------+
                 |
                 v
+----------------------------------+
| 03. DATA QUALITY                 |
| Schedule: After Silver           |
| Validate transformed data       |
| and business rules               |
+----------------------------------+
                 |
                 v
+----------------------------------+
| 04. REFERENTIAL INTEGRITY        |
| Schedule: After Data Quality     |
| Validate parent-child            |
| relationships                    |
+----------------------------------+
                 |
                 v
+----------------------------------+
| 05. GOLD REFRESH                 |
| Schedule: After Validation       |
| Refresh analytics and KPIs       |
+----------------------------------+
                 |
                 v
+----------------------------------+
| 06. FINAL VALIDATION             |
| Schedule: Last Task              |
| Verify pipeline outputs          |
| and execution status             |
+----------------------------------+
```

In [0]:

# ============================================================
# E-COMMERCE LAKEHOUSE
# FOLDER 11: DATABRICKS JOBS AND FINAL TESTING
# FILE 03: JOB SCHEDULING AND MONITORING
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

from datetime import datetime, timezone
import uuid


# ============================================================
# 2. CONFIGURATION
# ============================================================

CATALOG = "workspace"

GOLD_SCHEMA = "ecommerce_gold"

WORKFLOW_REPORT_TABLE = (
    "workspace.ecommerce_gold.workflow_execution_report"
)

MONITORING_TABLE = (
    "workspace.ecommerce_gold.job_monitoring_report"
)

JOB_HEALTH_TABLE = (
    "workspace.ecommerce_gold.job_health_summary"
)

SUMMARY_TASK_NAME = "pipeline_execution_summary"

SLOW_TASK_THRESHOLD_SECONDS = 300

MONITORING_RUN_ID = str(uuid.uuid4())

MONITORING_TIMESTAMP = (
    datetime.now(timezone.utc)
    .replace(tzinfo=None)
)

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS "
    f"{CATALOG}.{GOLD_SCHEMA}"
)

print("Configuration initialized.")


# ============================================================
# 3. CHECK WORKFLOW REPORT TABLE
# ============================================================

if not spark.catalog.tableExists(
    WORKFLOW_REPORT_TABLE
):

    raise Exception(
        "Workflow execution report does not exist. "
        "Execute File 02 before running monitoring."
    )


workflow_df = spark.table(
    WORKFLOW_REPORT_TABLE
)


# ============================================================
# 4. VALIDATE REQUIRED COLUMNS
# ============================================================

required_columns = [

    "run_id",
    "pipeline_name",
    "task_name",
    "pipeline_layer",
    "notebook_path",
    "execution_status",
    "start_timestamp",
    "end_timestamp",
    "duration_seconds",
    "error_message"

]

missing_columns = [

    column
    for column in required_columns
    if column not in workflow_df.columns

]

if missing_columns:

    raise ValueError(
        f"Missing workflow report columns: {missing_columns}"
    )


print("Workflow report schema validated.")


# ============================================================
# 5. IDENTIFY THE LATEST PIPELINE RUN
# ============================================================

# Use the pipeline summary record.
# Do NOT select the latest individual task timestamp.

summary_df = (

    workflow_df

    .filter(
        F.col("task_name") == SUMMARY_TASK_NAME
    )

)

if summary_df.limit(1).count() == 0:

    raise Exception(
        "No pipeline summary records found. "
        "The workflow report must contain "
        "pipeline_execution_summary records."
    )


summary_window = (

    Window

    .partitionBy("pipeline_name")

    .orderBy(

        F.col("start_timestamp").desc(),

        F.col("end_timestamp").desc()

    )

)


latest_pipeline_runs_df = (

    summary_df

    .withColumn(
        "run_rank",
        F.row_number().over(summary_window)
    )

    .filter(
        F.col("run_rank") == 1
    )

    .select(

        "pipeline_name",

        "run_id",

        F.col("start_timestamp").alias(
            "summary_start_timestamp"
        ),

        F.col("end_timestamp").alias(
            "summary_end_timestamp"
        ),

        F.col("execution_status").alias(
            "summary_execution_status"
        )

    )

)


print("\nLATEST PIPELINE RUNS")

display(
    latest_pipeline_runs_df
)


# ============================================================
# 6. GET ALL TASKS FOR THE LATEST RUNS
# ============================================================

# Join using pipeline_name and run_id.
# Exclude pipeline_execution_summary.

latest_execution_df = (

    workflow_df.alias("w")

    .join(

        latest_pipeline_runs_df.alias("l"),

        (

            F.col("w.pipeline_name")
            == F.col("l.pipeline_name")

        )

        &

        (

            F.col("w.run_id")
            == F.col("l.run_id")

        ),

        "inner"

    )

    .filter(

        F.col("w.task_name")
        != SUMMARY_TASK_NAME

    )

    .select(
        "w.*"
    )

)


if latest_execution_df.limit(1).count() == 0:

    raise Exception(
        "No task execution records found "
        "for the latest pipeline run."
    )


# Normalize execution status.

latest_execution_df = (

    latest_execution_df

    .withColumn(

        "normalized_status",

        F.upper(
            F.trim(
                F.col("execution_status")
            )
        )

    )

)


# ============================================================
# 7. REMOVE DUPLICATE TASK RECORDS
# ============================================================

# If a task has multiple records within the same run,
# retain the most recent execution record.

task_window = (

    Window

    .partitionBy(
        "pipeline_name",
        "run_id",
        "task_name"
    )

    .orderBy(

        F.col("start_timestamp").desc(),

        F.col("end_timestamp").desc()

    )

)


latest_execution_df = (

    latest_execution_df

    .withColumn(

        "task_rank",

        F.row_number().over(task_window)

    )

    .filter(
        F.col("task_rank") == 1
    )

    .drop(
        "task_rank"
    )

)


# ============================================================
# 8. TASK EXECUTION METRICS
# ============================================================

task_metrics_df = (

    latest_execution_df

    .groupBy(

        "pipeline_name",

        "run_id",

        "task_name",

        "pipeline_layer"

    )

    .agg(

        F.count("*").alias(
            "execution_count"
        ),

        F.sum(

            F.when(
                F.col("normalized_status") == "SUCCESS",
                1
            ).otherwise(0)

        ).alias(
            "successful_tasks"
        ),

        F.sum(

            F.when(
                F.col("normalized_status") == "FAILED",
                1
            ).otherwise(0)

        ).alias(
            "failed_tasks"
        ),

        F.sum(

            F.when(
                F.col("normalized_status") == "SKIPPED",
                1
            ).otherwise(0)

        ).alias(
            "skipped_tasks"
        ),

        F.avg(
            "duration_seconds"
        ).alias(
            "average_duration_seconds"
        ),

        F.max(
            "duration_seconds"
        ).alias(
            "maximum_duration_seconds"
        ),

        F.max(
            "start_timestamp"
        ).alias(
            "last_execution_timestamp"
        )

    )

)


# ============================================================
# 9. PIPELINE HEALTH SUMMARY
# ============================================================

pipeline_health_df = (

    latest_execution_df

    .groupBy(

        "pipeline_name",

        "run_id"

    )

    .agg(

        F.countDistinct(
            "task_name"
        ).alias(
            "total_tasks"
        ),

        F.sum(

            F.when(
                F.col("normalized_status") == "SUCCESS",
                1
            ).otherwise(0)

        ).alias(
            "successful_task_executions"
        ),

        F.sum(

            F.when(
                F.col("normalized_status") == "FAILED",
                1
            ).otherwise(0)

        ).alias(
            "failed_task_executions"
        ),

        F.sum(

            F.when(
                F.col("normalized_status") == "SKIPPED",
                1
            ).otherwise(0)

        ).alias(
            "skipped_task_executions"
        ),

        F.sum(

            F.when(

                ~F.col("normalized_status").isin(
                    "SUCCESS",
                    "FAILED",
                    "SKIPPED"
                ),

                1

            ).otherwise(0)

        ).alias(
            "other_task_executions"
        ),

        F.sum(
            "duration_seconds"
        ).alias(
            "total_task_execution_seconds"
        ),

        F.min(
            "start_timestamp"
        ).alias(
            "pipeline_start_timestamp"
        ),

        F.max(
            "end_timestamp"
        ).alias(
            "pipeline_end_timestamp"
        )

    )

)


# ============================================================
# 10. CALCULATE PIPELINE STATUS
# ============================================================

pipeline_health_df = (

    pipeline_health_df

    .withColumn(

        "pipeline_status",

        F.when(

            F.col("failed_task_executions") > 0,

            "FAILED"

        )

        .when(

            (

                F.col("skipped_task_executions") > 0

            )

            |

            (

                F.col("other_task_executions") > 0

            )

            |

            (

                F.col("successful_task_executions")
                != F.col("total_tasks")

            ),

            "INCOMPLETE"

        )

        .otherwise(
            "SUCCESS"
        )

    )

    .withColumn(

        "pipeline_duration_seconds",

        F.unix_timestamp(
            "pipeline_end_timestamp"
        )

        -

        F.unix_timestamp(
            "pipeline_start_timestamp"
        )

    )

    .withColumn(

        "task_success_rate",

        F.round(

            F.when(

                F.col("total_tasks") > 0,

                F.col("successful_task_executions")
                / F.col("total_tasks") * 100

            ).otherwise(0),

            2

        )

    )

    .drop(
        "skipped_task_executions",
        "other_task_executions"
    )

)


# ============================================================
# 11. FAILED TASK DETAILS
# ============================================================

failed_tasks_df = (

    latest_execution_df

    .filter(

        F.col("normalized_status") == "FAILED"

    )

    .select(

        "run_id",

        "pipeline_name",

        "task_name",

        "pipeline_layer",

        "notebook_path",

        "error_message",

        "start_timestamp",

        "end_timestamp",

        "duration_seconds"

    )

)


# ============================================================
# 12. SLOW TASK DETECTION
# ============================================================

slow_tasks_df = (

    latest_execution_df

    .filter(

        F.col("duration_seconds")
        > SLOW_TASK_THRESHOLD_SECONDS

    )

    .select(

        "run_id",

        "pipeline_name",

        "task_name",

        "pipeline_layer",

        "duration_seconds",

        "normalized_status"

    )

    .withColumn(

        "performance_status",

        F.lit("SLOW")

    )

)


# ============================================================
# 13. BUILD MONITORING REPORT
# ============================================================

monitoring_report_df = (

    latest_execution_df

    .withColumn(

        "monitoring_run_id",

        F.col("run_id")

    )

    .withColumn(

        "monitoring_timestamp",

        F.lit(
            MONITORING_TIMESTAMP
        ).cast("timestamp")

    )

    .withColumn(

        "monitoring_status",

        F.when(

            F.col("normalized_status") == "FAILED",

            "FAILURE"

        )

        .when(

            F.col("normalized_status") == "SKIPPED",

            "SKIPPED"

        )

        .when(

            F.col("duration_seconds")
            > SLOW_TASK_THRESHOLD_SECONDS,

            "SLOW"

        )

        .otherwise(
            "HEALTHY"
        )

    )

    .drop(
        "normalized_status"
    )

)


# ============================================================
# 14. DUPLICATE-SAFE MONITORING REPORT WRITE
# ============================================================

if not spark.catalog.tableExists(
    MONITORING_TABLE
):

    (

        monitoring_report_df

        .write

        .format("delta")

        .mode("overwrite")

        .saveAsTable(MONITORING_TABLE)

    )

    print("Monitoring report table created.")

else:

    monitoring_target_columns = set(

        spark.table(
            MONITORING_TABLE
        ).columns

    )

    monitoring_source_columns = set(
        monitoring_report_df.columns
    )

    if (
        monitoring_target_columns
        != monitoring_source_columns
    ):

        raise Exception(
            "Monitoring report schema mismatch. "
            "Inspect the existing table schema before "
            "performing a schema migration."
        )


    monitoring_delta = DeltaTable.forName(
        spark,
        MONITORING_TABLE
    )


    (

        monitoring_delta.alias("target")

        .merge(

            monitoring_report_df.alias("source"),

            """

            target.run_id = source.run_id

            AND

            target.pipeline_name = source.pipeline_name

            AND

            target.task_name = source.task_name

            AND

            target.monitoring_run_id = source.monitoring_run_id

            """

        )

        .whenMatchedUpdateAll()

        .whenNotMatchedInsertAll()

        .execute()

    )


print("Monitoring report saved successfully.")


# ============================================================
# 15. DUPLICATE-SAFE PIPELINE HEALTH WRITE
# ============================================================

health_output_df = (

    pipeline_health_df

    .withColumn(

        "monitoring_run_id",

        F.col("run_id")

    )

    .withColumn(

        "monitoring_timestamp",

        F.lit(
            MONITORING_TIMESTAMP
        ).cast("timestamp")

    )

)


if not spark.catalog.tableExists(
    JOB_HEALTH_TABLE
):

    (

        health_output_df

        .write

        .format("delta")

        .mode("overwrite")

        .saveAsTable(JOB_HEALTH_TABLE)

    )

    print("Job health summary table created.")

else:

    health_target_columns = set(

        spark.table(
            JOB_HEALTH_TABLE
        ).columns

    )

    health_source_columns = set(
        health_output_df.columns
    )

    if health_target_columns != health_source_columns:

        raise Exception(
            "Job health summary schema mismatch. "
            "Inspect the existing table schema before "
            "performing a schema migration."
        )


    health_delta = DeltaTable.forName(
        spark,
        JOB_HEALTH_TABLE
    )


    (

        health_delta.alias("target")

        .merge(

            health_output_df.alias("source"),

            """

            target.run_id = source.run_id

            AND

            target.pipeline_name = source.pipeline_name

            """

        )

        .whenMatchedUpdateAll()

        .whenNotMatchedInsertAll()

        .execute()

    )


print("Job health summary saved successfully.")


# ============================================================
# 16. DISPLAY TASK EXECUTION METRICS
# ============================================================

print("\n" + "=" * 70)

print("TASK EXECUTION METRICS")

print("=" * 70)

display(

    task_metrics_df

    .orderBy(
        "task_name"
    )

)


# ============================================================
# 17. DISPLAY PIPELINE HEALTH
# ============================================================

print("\n" + "=" * 70)

print("PIPELINE HEALTH SUMMARY")

print("=" * 70)

display(
    pipeline_health_df
)


# ============================================================
# 18. DISPLAY FAILED TASKS
# ============================================================

print("\n" + "=" * 70)

print("FAILED TASK DETAILS")

print("=" * 70)

display(
    failed_tasks_df
)


# ============================================================
# 19. DISPLAY SLOW TASKS
# ============================================================

print("\n" + "=" * 70)

print(
    f"SLOW TASKS "
    f"(Threshold: {SLOW_TASK_THRESHOLD_SECONDS} seconds)"
)

print("=" * 70)

display(
    slow_tasks_df
)


# ============================================================
# 20. FINAL MONITORING SUMMARY
# ============================================================

health_summary = (

    pipeline_health_df

    .select(

        "pipeline_name",

        "run_id",

        "pipeline_status",

        "total_tasks",

        "successful_task_executions",

        "failed_task_executions",

        "pipeline_duration_seconds",

        "task_success_rate"

    )

    .collect()

)


failed_count = failed_tasks_df.count()

slow_count = slow_tasks_df.count()


print("\n" + "=" * 70)

print("JOB MONITORING COMPLETED")

print("=" * 70)

print(
    f"Monitoring Run ID: {MONITORING_RUN_ID}"
)

print(
    f"Monitoring Report: {MONITORING_TABLE}"
)

print(
    f"Job Health Summary: {JOB_HEALTH_TABLE}"
)

print(
    f"Failed Tasks: {failed_count}"
)

print(
    f"Slow Tasks: {slow_count}"
)

for row in health_summary:

    print("\nPipeline:", row["pipeline_name"])

    print("Pipeline Run ID:", row["run_id"])

    print("Pipeline Status:", row["pipeline_status"])

    print("Total Tasks:", row["total_tasks"])

    print(
        "Successful Tasks:",
        row["successful_task_executions"]
    )

    print(
        "Failed Tasks:",
        row["failed_task_executions"]
    )

    print(
        "Pipeline Duration:",
        row["pipeline_duration_seconds"],
        "seconds"
    )

    print(
        "Task Success Rate:",
        row["task_success_rate"],
        "%"
    )


print("\n" + "=" * 70)



## SQL monitoring queries:


### Latest pipeline health

In [0]:
%sql

SELECT *
FROM workspace.ecommerce_gold.job_health_summary
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY pipeline_name
    ORDER BY monitoring_timestamp DESC
) = 1;


### Task failure analysis

In [0]:
%sql

SELECT
    task_name,
    pipeline_layer,
    COUNT(*) AS total_executions,
    SUM(
        CASE
            WHEN execution_status = 'FAILED'
            THEN 1 ELSE 0
        END
    ) AS failed_executions
FROM workspace.ecommerce_gold.job_monitoring_report
GROUP BY
    task_name,
    pipeline_layer
ORDER BY failed_executions DESC;


### Slow task identification

In [0]:
%sql

SELECT
    task_name,
    pipeline_layer,
    duration_seconds,
    monitoring_status
FROM workspace.ecommerce_gold.job_monitoring_report
WHERE monitoring_status = 'SLOW'
ORDER BY duration_seconds DESC;


### Pipeline execution duration

In [0]:
%sql

SELECT
    pipeline_name,
    run_id,
    pipeline_status,
    pipeline_duration_seconds,
    task_success_rate,
    monitoring_timestamp
FROM workspace.ecommerce_gold.job_health_summary
ORDER BY monitoring_timestamp DESC;